In [ ]:
pip install google-cloud-pipeline-components

In [ ]:
import pandas as pd
import numpy as np

# Generate sample customer churn data
np.random.seed(42)
n_samples = 1000

data = {
    'customer_id': range(1, n_samples + 1),
    'age': np.random.randint(18, 70, n_samples),
    'tenure_months': np.random.randint(1, 72, n_samples),
    'monthly_charges': np.random.uniform(20, 150, n_samples),
    'total_charges': np.random.uniform(100, 8000, n_samples),
    'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples),
    'payment_method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_samples),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples),
    'online_security': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
    'tech_support': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
}

# Create target variable (churn)
df = pd.DataFrame(data)
# Higher monthly charges and month-to-month contracts more likely to churn
churn_prob = (df['monthly_charges'] / 150) * 0.5 + (df['contract_type'] == 'Month-to-month') * 0.3
df['churn'] = (np.random.random(n_samples) < churn_prob).astype(int)

# Save to CSV
df.to_csv('customer_churn.csv', index=False)
print(f"Dataset created with {len(df)} rows")
print(f"Churn rate: {df['churn'].mean()*100:.2f}%")
print(df.head())

In [ ]:
from google.cloud import storage
import os

# Get your project ID automatically
PROJECT_ID = !gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]

REGION = "europe-west4"
BUCKET_NAME = ""  # Bucket names must be unique
PIPELINE_ROOT = ""
DATA_PATH = ""  # Path of the file

print(f"Project ID: {PROJECT_ID}")
print(f"Region: {REGION}")
print(f"Bucket: {BUCKET_NAME}")
print(f"Pipeline Root: {PIPELINE_ROOT}")
print(f"Data Path: {DATA_PATH}")

In [ ]:
from kfp import dsl, compiler
from google_cloud_pipeline_components.v1.automl.training_job import AutoMLTabularTrainingJobRunOp
from google_cloud_pipeline_components.v1.dataset import TabularDatasetCreateOp
from google_cloud_pipeline_components.v1.endpoint import EndpointCreateOp, ModelDeployOp

@dsl.pipeline(
    name="automl-churn-pipeline-new",
    description="AutoML pipeline for churn prediction"
)
def automl_churn_pipeline_new(
    project: str,
    location: str,
    display_name: str = "churn-automl-new"
):
    
    # Step 1: Create Vertex AI Dataset
    dataset_create_op = TabularDatasetCreateOp(
        project=project,
        location=location,
        display_name=f"{display_name}-dataset",
        gcs_source=f"gs://{BUCKET_NAME}/data/customer_churn.csv"
    )
    
    # Step 2: Train with AutoML
    training_job_op = AutoMLTabularTrainingJobRunOp(
        project=project,
        location=location,
        display_name=f"{display_name}-training",
        optimization_prediction_type="classification",
        optimization_objective="maximize-au-prc",
        dataset=dataset_create_op.outputs["dataset"],
        target_column="churn",
        budget_milli_node_hours=1000,  # 1 node hour
        disable_early_stopping=False
    )
     # Step 3: Create Endpoint
    endpoint_create_op = EndpointCreateOp(
        project=project,
        location=location,
        display_name=f"{display_name}-endpoint"
    )
    
    # Step 4: Deploy Model
    ModelDeployOp(
        model=training_job_op.outputs["model"],
        endpoint=endpoint_create_op.outputs["endpoint"],
        dedicated_resources_min_replica_count=1,
        dedicated_resources_machine_type="n1-standard-2"
    )
    # Compile
    compiler.Compiler().compile(
        pipeline_func=automl_churn_pipeline_new,
        package_path="automl_churn_pipeline.json"
    )

    print("✅ AutoML pipeline compiled!")

In [ ]:
from datetime import datetime
from google.cloud import aiplatform
aiplatform.init(project=PROJECT_ID, location=REGION)

job = aiplatform.PipelineJob(
    display_name=f"automl-churn-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
    template_path="automl_churn_pipeline.json",
    pipeline_root=f"gs://{BUCKET_NAME}/pipeline_root",
    parameter_values={
        "project": PROJECT_ID,
        "location": REGION,
        "display_name": "churn-automl"
    }
)

job.submit()

print("✅ AutoML pipeline submitted!")
print(f"This will take 1-2 hours (AutoML training time)")
print(f"\nView at: https://console.cloud.google.com/vertex-ai/pipelines")

In [ ]:
from google.cloud import aiplatform

# Setup
PROJECT_ID = !gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
REGION = "us-central1"

aiplatform.init(project=PROJECT_ID, location=REGION)

# Option 1: List ALL endpoints (simplest)
endpoints = aiplatform.Endpoint.list()

print(f"Found {len(endpoints)} endpoint(s):\n")
for i, endpoint in enumerate(endpoints, 1):
    print(f"{i}. {endpoint.display_name}")
    print(f"   ID: {endpoint.name}")
    print(f"   Created: {endpoint.create_time}")
    print()

# Get the latest endpoint
if endpoints:
    endpoint = endpoints[0]
    print(f"✅ Using endpoint: {endpoint.display_name}")
else:
    print("❌ No endpoints found")

In [ ]:
from google.cloud import aiplatform

# Get pipeline jobs
jobs = aiplatform.PipelineJob.list(order_by='create_time desc')

if jobs:
    latest_job = jobs[0]
    print(f"Latest Pipeline: {latest_job.display_name}")
    print(f"State: {latest_job.state.name}")
    
    print("\nTask Details:")
    for task in latest_job.task_details:
        print(f"\n  {task.task_name}")
        print(f"    State: {task.state.name}")
        
        if task.state.name == 'FAILED':
            print(f"    ❌ Error: {task.error}")

In [ ]:
from google.cloud import aiplatform

PROJECT_ID = !gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
REGION = "europe-west4"

aiplatform.init(project=PROJECT_ID, location=REGION)

# List models
print("Checking for trained models...")
models = aiplatform.Model.list(order_by='create_time desc')

if models:
    print(f"\n✅ Found {len(models)} model(s):\n")
    for i, model in enumerate(models, 1):
        print(f"{i}. {model.display_name}")
        print(f"   Created: {model.create_time}")
        print(f"   Resource: {model.resource_name}")
        print()
    
    latest_model = models[0]
    print(f"Latest model: {latest_model.display_name}")
else:
    print("❌ No models found")

# Check endpoints
print("\nChecking for endpoints...")
endpoints = aiplatform.Endpoint.list(order_by='create_time desc')

if endpoints:
    print(f"\n✅ Found {len(endpoints)} endpoint(s):\n")
    for i, ep in enumerate(endpoints, 1):
        print(f"{i}. {ep.display_name}")
        print(f"   Created: {ep.create_time}")
        print()
else:
    print("❌ No endpoints found - deployment didn't happen")

In [ ]:
from google.cloud import aiplatform

# Setup
PROJECT_ID = ""
REGION = "europe-west4"

aiplatform.init(project=PROJECT_ID, location=REGION)

# Get endpoint
endpoints = aiplatform.Endpoint.list(order_by='create_time desc')
endpoint = endpoints[0]

print(f"Using endpoint: {endpoint.display_name}\n")

# Test data with customer_id as STRING
test_customer = {
    "customer_id": "TEST_001",  # String, not number
    "age": 30,
    "tenure_months": 12,
    "monthly_charges": 85.0,
    "total_charges": 1020.0,
    "contract_type": "Month-to-month",
    "payment_method": "Electronic check",
    "internet_service": "Fiber optic",
    "online_security": "No",
    "tech_support": "No"
}

print("Test Customer Profile:")
print(f"  Age: {test_customer['age']}")
print(f"  Tenure: {test_customer['tenure_months']} months")
print(f"  Monthly Charges: ${test_customer['monthly_charges']:.2f}")
print(f"  Contract: {test_customer['contract_type']}")

print("\nMaking prediction...")

try:
    prediction = endpoint.predict(instances=[test_customer])
    
    print(f"\n✅ Prediction successful!")
    print(f"\n🔮 Result:")
    result = prediction.predictions[0]
    
    # Display raw result first
    print(f"Raw result: {result}")
    
    # Parse and display nicely
    if isinstance(result, dict):
        if 'classes' in result and 'scores' in result:
            predicted_class = result['classes'][0]
            confidence = result['scores'][0]
            print(f"\nPredicted Class: {predicted_class}")
            print(f"Confidence: {confidence:.2%}")
            
            if predicted_class == '1':
                print(f"\n⚠️  CUSTOMER LIKELY TO CHURN")
            else:
                print(f"\n✅ CUSTOMER LIKELY TO STAY")
        elif 'value' in result:
            print(f"\nPredicted Value: {result['value']}")
    elif isinstance(result, list):
        if len(result) == 2:
            prob_no_churn = result[0]
            prob_churn = result[1]
            print(f"\nProbability of staying: {prob_no_churn:.2%}")
            print(f"Probability of churning: {prob_churn:.2%}")
            
            if prob_churn > 0.5:
                print(f"\n⚠️  CUSTOMER LIKELY TO CHURN")
            else:
                print(f"\n✅ CUSTOMER LIKELY TO STAY")
        else:
            print(f"\nProbabilities: {result}")
    else:
        print(f"\nPrediction: {result}")
    
    print("\n" + "="*60)
    
except Exception as e:
    print(f"\n❌ Prediction failed: {e}")

In [ ]:
from google.cloud import aiplatform

PROJECT_ID = ""
REGION = "europe-west4"

aiplatform.init(project=PROJECT_ID, location=REGION)

endpoints = aiplatform.Endpoint.list(order_by='create_time desc')
endpoint = endpoints[0]

print("Testing string customer_id with different formats...\n")

test_formats = [
    {
        "name": "String with digits only: '1'",
        "data": {
            "age": 30.0,
            "contract_type": "Month-to-month",
            "total_charges": 1020.0,
            "tech_support": "No",
            "tenure_months": 12.0,
            "internet_service": "Fiber optic",
            "monthly_charges": 85.0,
            "customer_id": "1",  # String containing just a number
            "online_security": "No",
            "payment_method": "Electronic check"
        }
    },
    {
        "name": "String with digits: '9999'",
        "data": {
            "age": 30.0,
            "contract_type": "Month-to-month",
            "total_charges": 1020.0,
            "tech_support": "No",
            "tenure_months": 12.0,
            "internet_service": "Fiber optic",
            "monthly_charges": 85.0,
            "customer_id": "9999",  # String containing a number
            "online_security": "No",
            "payment_method": "Electronic check"
        }
    },
    {
        "name": "String with float format: '1.0'",
        "data": {
            "age": 30.0,
            "contract_type": "Month-to-month",
            "total_charges": 1020.0,
            "tech_support": "No",
            "tenure_months": 12.0,
            "internet_service": "Fiber optic",
            "monthly_charges": 85.0,
            "customer_id": "1.0",  # String containing float
            "online_security": "No",
            "payment_method": "Electronic check"
        }
    }
]

for test in test_formats:
    print(f"Testing: {test['name']}")
    print(f"  customer_id value: {test['data']['customer_id']} (type: {type(test['data']['customer_id']).__name__})")
    
    try:
        prediction = endpoint.predict(instances=[test['data']])
        print(f"  ✅ SUCCESS!")
        print(f"  Result: {prediction.predictions[0]}")
        print("\n" + "="*60 + "\n")
        
        # Parse and display the result nicely
        result = prediction.predictions[0]
        
        if isinstance(result, dict):
            if 'classes' in result and 'scores' in result:
                predicted_class = result['classes'][0]
                confidence = result['scores'][0]
                print(f"Predicted Class: {predicted_class}")
                print(f"Confidence: {confidence:.2%}")
                print(f"\n{'⚠️  LIKELY TO CHURN' if predicted_class == '1' else '✅ LIKELY TO STAY'}")
            elif 'value' in result:
                print(f"Predicted Value: {result['value']}")
        elif isinstance(result, list):
            if len(result) == 2:
                prob_no_churn = result[0]
                prob_churn = result[1]
                print(f"Probability of staying: {prob_no_churn:.2%}")
                print(f"Probability of churning: {prob_churn:.2%}")
                print(f"\n{'⚠️  LIKELY TO CHURN' if prob_churn > 0.5 else '✅ LIKELY TO STAY'}")
        
        print("\n" + "="*60)
        break  # Stop at first success
        
    except Exception as e:
        print(f"  ❌ Failed: {e}\n")

print("\nIf successful, the working format is customer_id as a string containing digits.")

In [ ]:
from google.cloud import aiplatform

PROJECT_ID = ""
REGION = "europe-west4"

aiplatform.init(project=PROJECT_ID, location=REGION)

endpoints = aiplatform.Endpoint.list(order_by='create_time desc')
endpoint = endpoints[0]

print("Testing with ALL fields as strings...\n")

# ALL values as strings (even numbers)
test_customer = {
    "age": "30",                           # String
    "contract_type": "Month-to-month",     # String
    "total_charges": "1020.0",             # String
    "tech_support": "No",                  # String
    "tenure_months": "12",                 # String
    "internet_service": "Fiber optic",     # String
    "monthly_charges": "85.0",             # String
    "customer_id": "9999",                 # String
    "online_security": "No",               # String
    "payment_method": "Electronic check"   # String
}

print("Test Customer Profile:")
print(f"  Age: {test_customer['age']}")
print(f"  Tenure: {test_customer['tenure_months']} months")
print(f"  Monthly Charges: ${test_customer['monthly_charges']}")
print(f"  Contract: {test_customer['contract_type']}")

print("\nAll fields are strings:")
for key, value in test_customer.items():
    print(f"  {key}: '{value}' (type: {type(value).__name__})")

print("\nMaking prediction...")

try:
    prediction = endpoint.predict(instances=[test_customer])
    
    print(f"\n✅ SUCCESS! Prediction received!")
    print("\n" + "="*60)
    
    result = prediction.predictions[0]
    print(f"Raw result: {result}")
    
    # Parse and display nicely
    if isinstance(result, dict):
        if 'classes' in result and 'scores' in result:
            predicted_class = result['classes'][0]
            confidence = result['scores'][0]
            print(f"\n🔮 Prediction:")
            print(f"  Predicted Class: {predicted_class}")
            print(f"  Confidence: {confidence:.2%}")
            
            if predicted_class == '1':
                print(f"\n  ⚠️  CUSTOMER LIKELY TO CHURN")
                print(f"  📊 Risk Level: HIGH")
            else:
                print(f"\n  ✅ CUSTOMER LIKELY TO STAY")
                print(f"  📊 Risk Level: LOW")
                
        elif 'value' in result:
            print(f"\nPredicted Value: {result['value']}")
            
    elif isinstance(result, list):
        if len(result) == 2:
            prob_no_churn = result[0]
            prob_churn = result[1]
            print(f"\n🔮 Prediction:")
            print(f"  Probability of staying: {prob_no_churn:.2%}")
            print(f"  Probability of churning: {prob_churn:.2%}")
            
            if prob_churn > 0.7:
                risk = "🔴 VERY HIGH"
            elif prob_churn > 0.5:
                risk = "🟠 HIGH"
            elif prob_churn > 0.3:
                risk = "🟡 MEDIUM"
            else:
                risk = "🟢 LOW"
            
            print(f"\n  Risk Level: {risk}")
            print(f"  Result: {'⚠️  LIKELY TO CHURN' if prob_churn > 0.5 else '✅ LIKELY TO STAY'}")
    else:
        print(f"\nPrediction: {result}")
    
    print("="*60)
    
except Exception as e:
    print(f"\n❌ Prediction failed: {e}")
    import traceback
    traceback.print_exc()